In [73]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from torch import nn


In [74]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size= 5, embeddings_dim = 8):
        super().__init__()
        self.fc1 = nn.Linear(vocab_size, embeddings_dim, bias=False)
        self.out = nn.Linear(embeddings_dim, vocab_size, bias=False)

    def forward(self, x):
        x = self.fc1(x)
        x = self.out(x)
        return x


In [75]:
def tokenize(text: str):
    tokens = np.array(text.lower().split()).reshape(-1,1)
    return tokens

In [76]:
def encoding(tokens):
    enc = OneHotEncoder(sparse_output=False)
    return enc.fit_transform(tokens)

In [77]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [78]:
phrase = encoding(tokenize("oi eu sou o thalles"))
print(phrase)

[[0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 1.]]


In [79]:
with open("data/text8") as f:
    text = f.read()

n_tokens = 20_000
corpus = text.split()[:n_tokens]
print(len(corpus), len(set(corpus)))

20000 4290


In [80]:
tokens = tokenize(" ".join(corpus))
encoded = encoding(tokens)
print(tokens.shape, encoded.shape)

(20000, 1) (20000, 4290)


In [81]:
model = SkipGram(vocab_size=encoded.shape[1], embeddings_dim=100).to(device)
print(model)

SkipGram(
  (fc1): Linear(in_features=4290, out_features=100, bias=False)
  (out): Linear(in_features=100, out_features=4290, bias=False)
)


In [82]:
X = torch.tensor(encoded, dtype=torch.float32).to(device)
ids = torch.tensor(encoded.argmax(axis=1)).to(device)
print(X.shape, ids.shape)

torch.Size([20000, 4290]) torch.Size([20000])


In [83]:
window = 2
centers, contexts = [], []
for i in range(len(tokens)):
    for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
        if j != i:
            centers.append(i)
            contexts.append(j)


In [84]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 512

for epoch in range(5):
    for i in range(0, len(centers), batch_size):
        x = X[centers[i:i + batch_size]]
        y = ids[contexts[i:i + batch_size]]

        loss = loss_fn(model(x), y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(epoch, loss.item())

0 8.352347373962402
1 8.076521873474121
2 7.848450183868408
3 7.633270740509033
4 7.447770118713379
